# A

```python

import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, root_mean_squared_log_error, r2_score
import json

import numpy as np

df = pd.read_csv("/content/train_weights.csv")

X = df.drop("MSE", axis=1)
y = np.log1p(df.MSE)

# Определение категориальных признаков
categorical_features_indices = np.where(X.dtypes != np.float64)[0]



model = CatBoostRegressor(iterations=2000,
                          learning_rate=0.03,
                          depth=6,
                          loss_function='RMSE',
                          verbose=False, # Отключаем вывод в процессе обучения
                          cat_features=categorical_features_indices)

# Обучение модели
model.fit(X, y)

# Предсказание на тестовых данных
predictions = model.predict(X)

# Оценка производительности модели
rmsle = root_mean_squared_log_error(y, predictions)
print(f"root_mean_squared_log_error: {rmsle}")
print(f"r2 {r2_score(y, predictions)}")
print(100*max(min((0.3 - rmsle) / 0.1, 1), 0))

X_test = pd.read_csv("/content/test_weights.csv")

predictions = model.predict(X_test)
X_test["MSE"] = np.expm1(predictions)

X_test.to_json('answers', orient='records')

with open('answers') as f:
  obja = json.load(f)

print(json.dumps(obja, indent=4))

```

# B

```python
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, classification_report
from catboost import CatBoostClassifier

df = pd.read_csv("/content/train.csv")
X = df.drop('target', axis=1)
y = df.target


cat_features_names = ['C']


model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.02,
    depth=6,
    loss_function='Logloss',  # Стандарт для бинарной классификации
    eval_metric='AUC',        # AUC - отличная метрика для бинарной классификации
    random_seed=42,
    verbose=100,              # Вывод прогресса каждые 100 итераций
    cat_features=cat_features_names # Указываем имена категориальных колонок
)

model.fit(X,y)

# Make predictions on the test set
y_pred = model.predict(X)

# Evaluate the model's performance
accuracy = accuracy_score(y, y_pred)
print(f"Accuracy: {accuracy:.4f}")

auc = roc_auc_score(y, y_pred)
print(f"AUC: {auc:.4f}")
print(f"Want points: {100*max(min((auc-0.8)/0.08,1),0)}")
print(confusion_matrix(y, y_pred))
print(classification_report(y, y_pred))

X_test = pd.read_csv("/content/test.csv")

pd.DataFrame(model.predict_proba(X_test)[:,1], columns=["target"]).to_csv("answers.csv", index=False)
```

# C

```cpp
#include <bits/stdc++.h>
using namespace std;
using pii = pair<int,int>;
const int INF = 1e9;

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    
    int n, m;
    if (!(cin >> n >> m)) return 0;
    int sx, sy;
    cin >> sx >> sy;
    --sx; --sy;
    vector<string> grid(n);
    for (int i = 0; i < n; ++i) cin >> grid[i];
    string s;
    cin >> s;
    if (s.empty()) { cout << 0 << '\n'; return 0; }

    // remove consecutive duplicates
    string ss;
    ss.push_back(s[0]);
    for (int i = 1; i < (int)s.size(); ++i)
        if (s[i] != ss.back()) ss.push_back(s[i]);
    s.swap(ss);

    int NM = n * m;
    auto idx = [&](int r, int c){ return r * m + c; };

    // locations as flat indices
    vector<vector<int>> loc(26);
    for (int r = 0; r < n; ++r)
        for (int c = 0; c < m; ++c)
            loc[grid[r][c] - 'a'].push_back(idx(r,c));

    // dp as flat array
    vector<int> dp(NM, INF), next_dp(NM, INF), trans(NM, INF);

    int first = s[0] - 'a';
    for (int p : loc[first]) {
        int r = p / m, c = p % m;
        dp[p] = abs(r - sx) + abs(c - sy);
    }

    auto manhattan_transform = [&](vector<int> &g) {
        // horizontal forward
        for (int r = 0; r < n; ++r) {
            int base = r * m;
            for (int c = 1; c < m; ++c) {
                int cur = base + c;
                int left = cur - 1;
                int nv = g[left] + 1;
                if (nv < g[cur]) g[cur] = nv;
            }
            for (int c = m - 2; c >= 0; --c) {
                int cur = base + c;
                int right = cur + 1;
                int nv = g[right] + 1;
                if (nv < g[cur]) g[cur] = nv;
            }
        }
        // vertical forward/back
        for (int c = 0; c < m; ++c) {
            for (int r = 1; r < n; ++r) {
                int cur = r * m + c;
                int up = cur - m;
                int nv = g[up] + 1;
                if (nv < g[cur]) g[cur] = nv;
            }
            for (int r = n - 2; r >= 0; --r) {
                int cur = r * m + c;
                int down = cur + m;
                int nv = g[down] + 1;
                if (nv < g[cur]) g[cur] = nv;
            }
        }
    };

    for (int i = 0; i + 1 < (int)s.size(); ++i) {
        int a = s[i] - 'a';
        int b = s[i+1] - 'a';
        const auto &A = loc[a];
        const auto &B = loc[b];

        // reset next_dp
        fill(next_dp.begin(), next_dp.end(), INF);

        long long prod = 1LL * A.size() * B.size();
        if (prod <= NM) {
            // brute pairwise
            for (int pa : A) {
                int valA = dp[pa];
                if (valA >= INF) continue;
                int rA = pa / m, cA = pa % m;
                for (int pb : B) {
                    int rB = pb / m, cB = pb % m;
                    int d = valA + abs(rA - rB) + abs(cA - cB);
                    if (d < next_dp[pb]) next_dp[pb] = d;
                }
            }
        } else {
            // transform approach
            // fill trans with INF and copy dp at A positions
            fill(trans.begin(), trans.end(), INF);
            for (int pa : A) {
                trans[pa] = dp[pa];
            }
            manhattan_transform(trans);
            for (int pb : B) next_dp[pb] = trans[pb];
        }
        dp.swap(next_dp);
    }

    int last = s.back() - 'a';
    int ans = INF;
    for (int p : loc[last]) if (dp[p] < ans) ans = dp[p];
    if (ans >= INF) ans = 0;
    cout << ans << '\n';
    return 0;
}

```

# D


```python
import sys
import math
sys.setrecursionlimit(10000)

def read_input():
    data = sys.stdin.read().strip().split()
    it = iter(data)
    t = int(next(it))
    tests = []
    for _ in range(t):
        n = int(next(it))
        edges = [[] for _ in range(n)]
        for _e in range(n-1):
            u = int(next(it)) - 1
            v = int(next(it)) - 1
            edges[u].append(v)
            edges[v].append(u)
        points = []
        for i in range(n):
            x = float(next(it)); y = float(next(it))
            points.append((x,y))
        tests.append((n, edges, points))
    return tests

def solve_one(n, edges, points):
    # Root the tree at 0
    parent = [-1]*n
    order = []
    stack = [0]
    parent[0] = -2  # mark root
    while stack:
        v = stack.pop()
        order.append(v)
        for to in edges[v]:
            if parent[to] == -1:
                parent[to] = v
                stack.append(to)
    parent[0] = -1

    # compute subtree sizes
    sz = [1]*n
    for v in reversed(order):
        p = parent[v]
        if p != -1:
            sz[p] += sz[v]

    # mapping vertex -> point index (0-based)
    assign = [-1]*n

    # initial set of all point indices
    all_pts = list(range(n))

    def rec(v, pts):
        """
        pts: list of point indices of length == sz[v]
        assign one of them to v, then partition remaining among children
        """
        # choose pivot: point with minimal x (then minimal y)
        best_idx = pts[0]
        bx, by = points[best_idx]
        for p in pts[1:]:
            x,y = points[p]
            if x < bx or (x == bx and y < by):
                best_idx = p
                bx, by = x, y

        assign[v] = best_idx

        # remaining points
        rem = [p for p in pts if p != best_idx]
        if not rem:
            return

        # compute angles of rem points around pivot
        angles = []
        px, py = points[best_idx]
        for p in rem:
            x,y = points[p]
            ang = math.atan2(y - py, x - px)
            angles.append((ang, p))
        angles.sort(key=lambda z: z[0])

        # children of v (exclude parent)
        children = [to for to in edges[v] if to != parent[v]]
        # for determinism, keep children order as is (any order works)
        # partition angles into blocks for children by subtree sizes
        idx = 0
        for child in children:
            need = sz[child]
            block = [p for _,p in angles[idx: idx + need]]
            idx += need
            rec(child, block)

        # leaf or done
        return

    rec(0, all_pts)

    # produce permutation p_1..p_n where p_i is 1-based index of point assigned to vertex i
    perm = [assign[i] + 1 for i in range(n)]
    return perm

def main():
    tests = read_input()
    out_lines = []
    for (n, edges, points) in tests:
        perm = solve_one(n, edges, points)
        out_lines.append(" ".join(map(str, perm)))
    print("\n".join(out_lines))

if __name__ == "__main__":
    main()
```